# 建立影像生成應用程式 (Azure OpenAI)

LLM 不僅能生成文字，還能從文字描述生成影像。影像作為一種模態，廣泛應用於醫療科技、建築、旅遊、遊戲開發、行銷等領域。在本課中，我們將使用 Microsoft Foundry 上的當代 **GPT 影像** 模型。

## 學習目標

本課程結束時您將能夠：

- 解釋什麼是影像生成及其應用場景。
- 了解 `gpt-image` 模型家族及其與傳統 DALL·E 模型的區別。
- 建立影像生成應用程式並進行影像編輯。

## 什麼是影像生成？

影像生成模型透過文字提示創建影像。現代的模型如 `gpt-image` 在訓練期間學習文字與影像間的關聯，接著迭代地將隨機噪音轉換成符合提示內容的影像。

兩個知名的影像模型家族有：

- **`gpt-image` (OpenAI)** — 本課程使用的當代世代模型。支援文字轉影像生成及影像編輯（帶遮罩的圖像修補）。
- **Midjourney** — 一個受歡迎的第三方模型，擁有自己的服務及以 Discord 為基礎的工作流程。

> 舊版 OpenAI 影像模型 — **DALL·E 2** 與 **DALL·E 3** — 屬於傳統模型。DALL·E 3 不再提供新的部署，且如 `create_variation` 等功能僅存在於 DALL·E 2。新應用請使用 `gpt-image` 模型。

在 Microsoft Foundry 上，**`gpt-image-2`** 是最新且功能最強大的影像模型，建議作為預設使用。`gpt-image-1.5` 和 `gpt-image-1-mini` 也均可使用。

> **重要提示：** `gpt-image` 模型會回傳以 **base64** (`b64_json`) 編碼的生成影像，而非影像 URL。您的程式碼需將 base64 字串解碼為位元組並儲存，沒有可下載的影像 URL。


## 建立您的第一個圖像生成應用程式

那麼建立一個圖像生成應用程式需要什麼呢？您需要以下庫：

- **python-dotenv**，強烈建議您使用此庫將您的秘密資訊保存在 *.env* 檔案中，遠離程式碼。
- **openai**，此庫是您與 OpenAI API 交互時會使用的。
- **pillow**，用於在 Python 中處理圖像。

如果尚未完成，請按照 [Microsoft Learn](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/create-resource?pivots=web-portal&WT.mc_id=academic-105485-koreyst) 頁面上的說明建立 Microsoft Foundry 資源和模型。選擇 **gpt-image-2** 作為模型（最新的 Azure OpenAI 圖像模型；DALL·E 是舊版）。

1. 建立一個 *.env* 檔案，內容如下：

    ```text
    AZURE_OPENAI_ENDPOINT=<your endpoint>
    AZURE_OPENAI_API_KEY=<your key>
    AZURE_OPENAI_DEPLOYMENT="gpt-image-2"
    ```

    在 Microsoft Foundry 入口網站的「部署」部分找到這些資訊。



1. 將上述函式庫收集到一個名為 *requirements.txt* 的檔案中，如下所示：

    ```text
    python-dotenv
    openai
    pillow
    ```


1. 接著，建立虛擬環境並安裝這些函式庫：


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!NOTE]
> 對於 Windows，請使用以下命令來建立並啟動您的虛擬環境：

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. 在名為 *app.py* 的檔案中新增以下程式碼：

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    from openai import AzureOpenAI, BadRequestError

    # 載入 dotenv
    dotenv.load_dotenv()

    # 配置 Azure OpenAI (Microsoft Foundry) 客戶端。
    # 影像模型需要較新的 API 版本 — 請查看 Foundry 文件以確認您的模型所需版本。
    client = AzureOpenAI(
      azure_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"],
      api_key=os.environ['AZURE_OPENAI_API_KEY'],
      api_version = "2025-04-01-preview"
      )

    try:
        # 使用影像生成 API 創建影像
        generation_response = client.images.generate(
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # 在此輸入您的提示文字
            size='1024x1024',
            n=1,
            model=os.environ['AZURE_OPENAI_DEPLOYMENT']   # 例如 "gpt-image-2"
        )
        # 設定存放影像的目錄
        image_dir = os.path.join(os.curdir, 'images')

        # 如果目錄不存在，則創建它
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # 初始化影像路徑（請注意檔案類型應為 png）
        image_path = os.path.join(image_dir, 'generated-image.png')

        # gpt-image 模型回傳影像為 base64 (b64_json) 格式，而非 URL
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # 在預設影像檢視器中顯示影像
        image = Image.open(image_path)
        image.show()

    # 捕捉例外狀況
    except BadRequestError as err:
        print(err)

    ```

讓我們解釋這段程式碼：

- 首先，我們匯入所需的函式庫，包括 OpenAI 函式庫、dotenv 函式庫、base64 模組以及 Pillow 函式庫。

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    from openai import AzureOpenAI, BadRequestError
    ```

- 接著，我們從 *.env* 檔案載入環境變數。

    ```python
    # 匯入 dotenv
    dotenv.load_dotenv()
    ```

- 之後，我們設定 Azure OpenAI（Microsoft Foundry）用戶端。

    ```python
    client = AzureOpenAI(
      azure_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"],
      api_key=os.environ['AZURE_OPENAI_API_KEY'],
      api_version = "2025-04-01-preview"
      )
    ```

- 接著，我們產生圖片：

    ```python
    generation_response = client.images.generate(
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # 在此輸入您的提示文字
        size='1024x1024',
        n=1,
        model=os.environ['AZURE_OPENAI_DEPLOYMENT']
    )
    ```

    `gpt-image` 模型會以 **base64** 字串形式回傳圖片，位於 `data[0].b64_json`。我們將它解碼為位元組並寫入檔案 — 並沒有下載用的 URL。

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- 最後，我們開啟圖片並使用標準圖片檢視器來顯示它：

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### 關於生成圖片的更多細節

來看看 `images.generate` 的參數：

- **prompt** 是用來生成圖片的文字提示。此處是「Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils」。
- **size** 是生成圖片的尺寸（例如 `1024x1024`、`1536x1024`、`1024x1536`，或 `"auto"`）。
- **n** 是要生成的圖片數量。這裡生成一張。
- **model** 是您的圖片模型部署名稱（例如 `gpt-image-2`）。

> 圖片模型不接受 `temperature` 參數 — 那是文字生成的控制。若要獲得多樣化，請再次呼叫 API；若要減少多樣化，請讓提示語更具體。

## 圖片生成的額外功能

您已經看到如何用幾行 Python 生成圖片。`gpt-image` 模型還可以 <strong>編輯</strong> 現有圖片。透過提供圖片、選擇性的 <strong>遮罩</strong>（標記要改變的區域）和提示語，您可以修改圖片的部分內容 — 例如，為兔子加戴一頂帽子。

```python
response = client.images.edit(
  model=os.environ['AZURE_OPENAI_DEPLOYMENT'],
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# 編輯內容也會以 base64 回傳
edited_image = base64.b64decode(response.data[0].b64_json)
```

基本圖片只有兔子；最終圖片中兔子戴上了帽子。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
此文件已使用 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 進行翻譯。雖然我們努力追求準確性，但請注意自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應視為權威來源。對於關鍵資訊，建議採用專業人工翻譯。我們不對因使用此翻譯所產生的任何誤解或誤譯承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
